In [ ]:

def build_feature_cols(df_features):
    """
    Selects the modelling feature columns from the full feature DataFrame,
    attaches the binary Target label, computes point-biserial correlations
    with the target, and returns:

        feature_df  : DataFrame containing only feature_cols + Target
        corr_df     : correlation summary sorted by |correlation|

    Parameters
    ----------
    df_features : pd.DataFrame
        Output of build_features().

    Returns
    -------
    feature_df : pd.DataFrame
    corr_df    : pd.DataFrame
    """
    feature_cols = [
        # Short-term momentum / mean-reversion
        'return_lag1', 'return_lag2', 'return_lag3', 'return_lag5',
        # Medium-term returns
        'return_5d', 'return_10d', 'return_20d',
        # Momentum oscillators
        'RSI_14', 'RSI_7', 'RSI_vs_50',
        # Trend / momentum
        'MACD', 'MACD_hist',
        # Bollinger band position and width
        'BB_position', 'BB_width',
        # Volatility
        'vol_ratio', 'ATR_ratio',
        # Volume
        'vol_ratio_20', 'OBV_change', 'VWAP_ratio',
        # Price vs. moving averages
        'price_vs_SMA10', 'price_vs_SMA50', 'SMA10_vs_SMA50',
        # Intraday structure
        'hl_range', 'oc_range', 'lower_wick', 'upper_wick',
        # Market / relative strength
        'spy_return_1d', 'rel_strength_5d',
        # Calendar effects
        'day_of_week', 'month',
    ]

    # Binary target: 1 if next-day close is higher, else 0
    target = (df_features['Close'].shift(-1) > df_features['Close']).astype(int)

    # Slice to feature columns + target, drop rows where any feature is NaN
    feature_df = df_features[feature_cols].copy()
    feature_df['Target'] = target
    feature_df = feature_df.dropna()


In [1]:
# ============================================================
# PHASE 3 DEEP DIVE: Comprehensive feature engineering
# ============================================================

import numpy as np
import pandas as pd
from scipy import stats


def build_features(df_input, spy_input):
    """
    Master feature engineering function.
    Takes raw OHLCV data and returns a rich feature DataFrame.
    Every feature is documented with its financial intuition.
    """
    df = df_input.copy()
    spy = spy_input.copy()

    # ── RETURNS (the foundation) ─────────────────────────────────
    df['return_1d']  = df['Close'].pct_change(1)
    df['return_3d']  = df['Close'].pct_change(3)
    df['return_5d']  = df['Close'].pct_change(5)
    df['return_10d'] = df['Close'].pct_change(10)
    df['return_20d'] = df['Close'].pct_change(20)

    # Lagged returns: short-term momentum and mean-reversion signals
    for lag in [1, 2, 3, 5]:
        df[f'return_lag{lag}'] = df['return_1d'].shift(lag)

    # ── TREND INDICATORS ─────────────────────────────────────────
    for window in [5, 10, 20, 50, 200]:
        df[f'SMA_{window}'] = df['Close'].rolling(window).mean()

    df['price_vs_SMA10']  = df['Close'] / df['SMA_10']  - 1
    df['price_vs_SMA50']  = df['Close'] / df['SMA_50']  - 1
    df['price_vs_SMA200'] = df['Close'] / df['SMA_200'] - 1
    df['SMA10_vs_SMA50']  = df['SMA_10'] / df['SMA_50'] - 1

    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()

    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']

    # ── MOMENTUM INDICATORS ──────────────────────────────────────
    def compute_rsi(series, period=14):
        delta = series.diff()
        gain  = delta.clip(lower=0).rolling(period).mean()
        loss  = (-delta.clip(upper=0)).rolling(period).mean()
        rs    = gain / loss.replace(0, np.nan)
        return 100 - (100 / (1 + rs))

    df['RSI_14']        = compute_rsi(df['Close'], 14)
    df['RSI_7']         = compute_rsi(df['Close'], 7)
    df['RSI_14_change'] = df['RSI_14'].diff()
    df['RSI_vs_50']     = df['RSI_14'] - 50

    df['ROC_5']  = df['Close'].pct_change(5)  * 100
    df['ROC_20'] = df['Close'].pct_change(20) * 100

    # ── VOLATILITY INDICATORS ─────────────────────────────────────
    df['vol_5d']  = df['return_1d'].rolling(5).std()  * np.sqrt(252)
    df['vol_20d'] = df['return_1d'].rolling(20).std() * np.sqrt(252)
    df['vol_60d'] = df['return_1d'].rolling(60).std() * np.sqrt(252)
    df['vol_ratio'] = df['vol_5d'] / df['vol_20d']

    bb_mid = df['Close'].rolling(20).mean()
    bb_std = df['Close'].rolling(20).std()
    df['BB_upper']    = bb_mid + 2 * bb_std
    df['BB_lower']    = bb_mid - 2 * bb_std
    df['BB_width']    = (df['BB_upper'] - df['BB_lower']) / bb_mid
    df['BB_position'] = (df['Close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])

    df['TR'] = np.maximum(
        df['High'] - df['Low'],
        np.maximum(
            abs(df['High'] - df['Close'].shift(1)),
            abs(df['Low']  - df['Close'].shift(1))
        )
    )
    df['ATR_14']    = df['TR'].rolling(14).mean()
    df['ATR_ratio'] = df['ATR_14'] / df['Close']

    # ── VOLUME INDICATORS ─────────────────────────────────────────
    df['vol_ratio_20'] = df['Volume'] / df['Volume'].rolling(20).mean()

    obv = [0]
    for i in range(1, len(df)):
        if df['Close'].iloc[i] > df['Close'].iloc[i-1]:
            obv.append(obv[-1] + df['Volume'].iloc[i])
        elif df['Close'].iloc[i] < df['Close'].iloc[i-1]:
            obv.append(obv[-1] - df['Volume'].iloc[i])
        else:
            obv.append(obv[-1])
    df['OBV']        = obv
    df['OBV_change'] = df['OBV'].pct_change(5)

    df['VWAP_ratio'] = df['Close'] / (
        (df['Close'] * df['Volume']).rolling(20).sum() /
        df['Volume'].rolling(20).sum()
    )

    # ── INTRADAY FEATURES ─────────────────────────────────────────
    df['hl_range']   = (df['High'] - df['Low'])  / df['Close']
    df['oc_range']   = (df['Close'] - df['Open']) / df['Open']
    df['upper_wick'] = (df['High'] - np.maximum(df['Open'], df['Close'])) / df['Close']
    df['lower_wick'] = (np.minimum(df['Open'], df['Close']) - df['Low'])  / df['Close']

    # ── MARKET FEATURES (SPY as market proxy) ─────────────────────
    spy['spy_return_1d'] = spy['Close'].pct_change(1)
    spy['spy_return_5d'] = spy['Close'].pct_change(5)
    df['spy_return_1d']  = spy['spy_return_1d'].reindex(df.index)
    df['spy_return_5d']  = spy['spy_return_5d'].reindex(df.index)
    df['rel_strength_5d'] = df['return_5d'] - df['spy_return_5d']

    # ── CALENDAR FEATURES ─────────────────────────────────────────
    df['day_of_week']    = df.index.dayofweek
    df['month']          = df.index.month
    df['quarter']        = df.index.quarter
    df['is_month_end']   = df.index.is_month_end.astype(int)
    df['is_quarter_end'] = df.index.is_quarter_end.astype(int)

    return df


def build_feature_cols(df_features):
    """
    Selects the modelling feature columns from the full feature DataFrame,
    attaches the binary Target label, computes point-biserial correlations
    with the target, and returns:

        feature_df  : DataFrame containing only feature_cols + Target
        corr_df     : correlation summary sorted by |correlation|

    Parameters
    ----------
    df_features : pd.DataFrame
        Output of build_features().

    Returns
    -------
    feature_df : pd.DataFrame
    corr_df    : pd.DataFrame
    """
    feature_cols = [
        # Short-term momentum / mean-reversion
        'return_lag1', 'return_lag2', 'return_lag3', 'return_lag5',
        # Medium-term returns
        'return_5d', 'return_10d', 'return_20d',
        # Momentum oscillators
        'RSI_14', 'RSI_7', 'RSI_vs_50',
        # Trend / momentum
        'MACD', 'MACD_hist',
        # Bollinger band position and width
        'BB_position', 'BB_width',
        # Volatility
        'vol_ratio', 'ATR_ratio',
        # Volume
        'vol_ratio_20', 'OBV_change', 'VWAP_ratio',
        # Price vs. moving averages
        'price_vs_SMA10', 'price_vs_SMA50', 'SMA10_vs_SMA50',
        # Intraday structure
        'hl_range', 'oc_range', 'lower_wick', 'upper_wick',
        # Market / relative strength
        'spy_return_1d', 'rel_strength_5d',
        # Calendar effects
        'day_of_week', 'month',
    ]

    # Binary target: 1 if next-day close is higher, else 0
    target = (df_features['Close'].shift(-1) > df_features['Close']).astype(int)

    # Slice to feature columns + target, drop rows where any feature is NaN
    feature_df = df_features[feature_cols].copy()
    feature_df['Target'] = target
    feature_df = feature_df.dropna()

    # ── CORRELATION ANALYSIS ──────────────────────────────────────
    correlations = {}
    for col in feature_cols:
        clean = feature_df[[col, 'Target']].dropna()
        corr, pval = stats.pointbiserialr(clean[col], clean['Target'])
        correlations[col] = {
            'correlation': corr,
            'p_value':     pval,
            'significant': pval < 0.05,
        }

    corr_df = (
        pd.DataFrame(correlations)
        .T
        .sort_values('correlation', key=abs, ascending=False)
        .astype({'correlation': float, 'p_value': float, 'significant': bool})
    )

    print(f"Feature matrix : {feature_df.shape[0]} rows × {feature_df.shape[1]} columns")
    print(f"Target balance : {feature_df['Target'].mean():.2%} up-days\n")
    print("Top 15 features by |correlation| with target:")
    print(corr_df.head(15).round(4).to_string())

    return feature_df, corr_df


def save_features_parquet(feature_df, corr_df,
                          features_path="features.parquet",
                          corr_path="feature_correlations.parquet"):
    """
    Persists both DataFrames to Parquet.

    Parameters
    ----------
    feature_df     : pd.DataFrame  — output of build_feature_cols()
    corr_df        : pd.DataFrame  — correlation summary
    features_path  : str           — destination for the feature matrix
    corr_path      : str           — destination for the correlation table
    """
    feature_df.to_parquet(features_path, index=True, engine="pyarrow")
    corr_df.to_parquet(corr_path,    index=True, engine="pyarrow")
    print(f"Saved feature matrix  → {features_path}  ({feature_df.shape})")
    print(f"Saved correlation table → {corr_path}  ({corr_df.shape})")


# ── USAGE ─────────────────────────────────────────────────────────────────────

# Step 1 – build the full feature set
df_features = build_features(raw_data["AAPL"], raw_data["SPY"])
print(f"Full feature set: {df_features.shape[1]} columns, {df_features.shape[0]} rows\n")

# Step 2 – select modelling columns, compute correlations
feature_df, corr_df = build_feature_cols(df_features)

# Step 3 – persist to Parquet
save_features_parquet(
    feature_df,
    corr_df,
    features_path="aapl_features.parquet",
    corr_path="aapl_feature_correlations.parquet",
)

NameError: name 'raw_data' is not defined

In [ ]:
# Step 1 – build the full feature set
df_features = build_features(raw_data["AAPL"], raw_data["SPY"])
print(f"Full feature set: {df_features.shape[1]} columns, {df_features.shape[0]} rows\n")

# Step 2 – select modelling columns, compute correlations
feature_df, corr_df = build_feature_cols(df_features)

# Step 3 – persist to Parquet
save_features_parquet(
    feature_df,
    corr_df,
    features_path="aapl_features.parquet",
    corr_path="aapl_feature_correlations.parquet",
)

In [ ]:
import yfinance as yf

# ── DOWNLOAD RAW DATA ─────────────────────────────────────────────────────────
tickers = ["AAPL", "SPY"]
raw_data = {}

for ticker in tickers:
    raw_data[ticker] = yf.download(ticker, start="2018-01-01", end="2024-12-31", auto_adjust=True)
    raw_data[ticker].columns = raw_data[ticker].columns.get_level_values(0)  # flatten MultiIndex if present
    print(f"{ticker}: {raw_data[ticker].shape[0]} rows downloaded")

In [ ]:
"""
feature_cols = [
    'return_lag1', 'return_lag2', 'return_lag3', 'return_lag5',
    'return_5d', 'return_10d', 'return_20d',
    'RSI_14', 'RSI_7', 'RSI_vs_50',
    'MACD', 'MACD_hist',
    'BB_position', 'BB_width',
    'vol_ratio', 'ATR_ratio',
    'vol_ratio_20', 'OBV_change', 'VWAP_ratio',
    'price_vs_SMA10', 'price_vs_SMA50', 'SMA10_vs_SMA50',
    'hl_range', 'oc_range', 'lower_wick', 'upper_wick',
    'spy_return_1d', 'rel_strength_5d',
    'day_of_week', 'month'
]

df_features['Target'] = (df_features['Close'].shift(-1) > df_features['Close']).astype(int)

# Compute point-biserial correlation of each feature with the binary target

correlations = {}
for col in feature_cols:
    clean = df_features[[col, 'Target']].dropna()
    corr, pval = stats.pointbiserialr(clean[col], clean['Target'])
    correlations[col] = {'correlation': corr, 'p_value': pval, 'significant': pval < 0.05}

corr_df = pd.DataFrame(correlations).T.sort_values('correlation', key=abs, ascending=False)
print("\nTop 15 features by correlation with target:")
print(corr_df.head(15).round(4))
"""

In [ ]:
# ============================================================
# PHASE 6 DEEP DIVE: SHAP explainability + production pipeline
# ============================================================
import shap

# Extract the raw XGBoost model from the tuned pipeline
best_model  = best_xgb.named_steps['model']
best_scaler = best_xgb.named_steps['scaler']

X_holdout_scaled = best_scaler.transform(X_holdout)
X_holdout_df     = pd.DataFrame(X_holdout_scaled, columns=feature_cols)

# Build SHAP explainer
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_holdout_df)

# --- Global explanation: what does the model rely on overall? ---
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_holdout_df, feature_names=feature_cols,
                  plot_type="bar", show=False)
plt.title("Global Feature Importance (mean |SHAP value|)")
plt.tight_layout(); plt.show()

# --- Detailed SHAP plot: how does each feature VALUE affect predictions? ---
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_holdout_df, feature_names=feature_cols, show=False)
plt.title("SHAP Summary: Feature Values vs Prediction Impact")
plt.tight_layout(); plt.show()

In [ ]:
# --- Local explanation: why did the model make THIS specific prediction? ---
# Pick an interesting day — e.g. the largest wrong prediction
wrong_idx = np.where(tuned_preds != y_holdout.values)[0]
sample_idx = wrong_idx[0]  # First wrong prediction

print(f"\nExplaining prediction for: {X_holdout.index[sample_idx].date()}")
print(f"Model predicted: {'Up' if tuned_preds[sample_idx] == 1 else 'Down'}")
print(f"Actual outcome:  {'Up' if y_holdout.iloc[sample_idx] == 1 else 'Down'}")
print(f"Confidence (proba): {tuned_proba[sample_idx]:.3f}")

shap.force_plot(
    explainer.expected_value,
    shap_values[sample_idx],
    X_holdout_df.iloc[sample_idx],
    feature_names=feature_cols,
    matplotlib=True,
    show=True
)

In [ ]:
# --- Production-ready prediction pipeline ---
# This is how you'd use the model in a live system.

class QuantPredictionPipeline:
    """
    A production-ready wrapper around our trained ML model.
    Given a stock ticker, it downloads the latest data,
    engineers features, and returns a trading signal.
    """
    def __init__(self, model, scaler, feature_cols, spy_ticker="SPY"):
        self.model        = model
        self.scaler       = scaler
        self.feature_cols = feature_cols
        self.spy_ticker   = spy_ticker

    def get_latest_signal(self, ticker, lookback_days=300):
        """
        Downloads fresh data and returns:
        - signal: 1 (go long) or 0 (stay out)
        - confidence: probability of an up day
        - top_drivers: the 3 features pushing the prediction
        """
        # Download fresh data
        end_date   = pd.Timestamp.today()
        start_date = end_date - pd.Timedelta(days=lookback_days)

        raw    = yf.download(ticker,     start=start_date, end=end_date,
                             auto_adjust=True, progress=False)
        spy    = yf.download(self.spy_ticker, start=start_date, end=end_date,
                             auto_adjust=True, progress=False)

        if len(raw) < 60:
            raise ValueError(f"Not enough data for {ticker}")

        # Engineer features
        featured = build_features(raw, spy)

        # Get the most recent complete row
        latest = featured[self.feature_cols].dropna().iloc[[-1]]

        # Scale and predict
        latest_scaled = self.scaler.transform(latest)
        proba  = self.model.predict_proba(latest_scaled)[0, 1]
        signal = int(proba > 0.5)

        # SHAP explanation for this prediction
        exp          = shap.TreeExplainer(self.model)
        shap_vals    = exp.shap_values(pd.DataFrame(latest_scaled, columns=self.feature_cols))
        top_idx      = np.argsort(np.abs(shap_vals[0]))[::-1][:3]
        top_drivers  = [(self.feature_cols[i], round(shap_vals[0][i], 4)) for i in top_idx]

        return {
            'ticker':     ticker,
            'date':       latest.index[0].date(),
            'signal':     signal,
            'confidence': round(proba, 4),
            'action':     "BUY / HOLD LONG" if signal == 1 else "STAY OUT / FLAT",
            'top_drivers': top_drivers
        }


# Instantiate and run the pipeline
pipeline = QuantPredictionPipeline(
    model        = best_model,
    scaler       = best_scaler,
    feature_cols = feature_cols
)

result = pipeline.get_latest_signal("AAPL")

print("\n" + "="*50)
print(f"  LIVE PREDICTION for {result['ticker']}")
print("="*50)
print(f"  Date:       {result['date']}")
print(f"  Signal:     {result['signal']}")
print(f"  Action:     {result['action']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"\n  Top prediction drivers:")
for feat, shap_val in result['top_drivers']:
    direction = "▲ pushes UP" if shap_val > 0 else "▼ pushes DOWN"
    print(f"    {feat:25s}  {direction}  (SHAP: {shap_val:+.4f})")
print("="*50)

In [ ]:
# ============================================================
# PHASE 6 DEEP DIVE: SHAP explainability + production pipeline
# ============================================================

# ── STEP 2: TRAIN / HOLDOUT SPLIT (time-aware) ───────────────
feature_cols = [col for col in df_features.columns if col != 'Target']

# Recreate Target from return_1d (next-day direction)
df_features['Target'] = (df_features['return_1d'].shift(-1) > 0).astype(int)
df_features = df_features.dropna()

X = df_features[feature_cols]
y = df_features['Target']

split_idx = int(len(X) * 0.80)

X_train,  X_holdout = X.iloc[:split_idx],  X.iloc[split_idx:]
y_train,  y_holdout = y.iloc[:split_idx],  y.iloc[split_idx:]

print(f"Train  : {X_train.shape[0]} rows  ({X_train.index[0].date()} → {X_train.index[-1].date()})")
print(f"Holdout: {X_holdout.shape[0]} rows  ({X_holdout.index[0].date()} → {X_holdout.index[-1].date()})")




# ── STEP 3b: HYPERPARAMETER TUNING + PIPELINE ────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', xgb.XGBClassifier(
        eval_metric='logloss',
        random_state=42
    ))
])

param_dist = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 4, 5],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.7, 0.8, 0.9],
    'model__colsample_bytree': [0.7, 0.8, 0.9],
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

search.fit(X_train, y_train)

best_xgb = search.best_estimator_  # ← defines best_xgb

print(f"Best params : {search.best_params_}")
print(f"Best AUC    : {search.best_score_:.4f}")

# ── STEP 4: PREDICT WITH TUNED PIPELINE ──────────────────────
tuned_proba = best_xgb.predict_proba(X_holdout)[:, 1]
tuned_preds = (tuned_proba > 0.5).astype(int)  # ← defines tuned_preds

print(f"Predicted proba — mean: {tuned_proba.mean():.3f}  "
      f"min: {tuned_proba.min():.3f}  "
      f"max: {tuned_proba.max():.3f}")

# ── STEP 8: SHAP — GLOBAL EXPLAINABILITY ─────────────────────
import shap

# Extract components from tuned pipeline
best_model = best_xgb.named_steps['model']
best_scaler = best_xgb.named_steps['scaler']

# Scale holdout and preserve dates + feature names
X_holdout_scaled = best_scaler.transform(X_holdout)
X_holdout_df = pd.DataFrame(
    X_holdout_scaled,
    columns=feature_cols,
    index=X_holdout.index
)

# Build SHAP explainer
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_holdout_df)

print(f"SHAP values computed — shape: {shap_values.shape}")
print(f"Explaining {X_holdout_df.shape[0]} holdout predictions "
      f"across {X_holdout_df.shape[1]} features\n")

# Plot 1: Global feature importance (mean |SHAP|)
# Answers: which features does the model rely on most overall?
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_holdout_df,
    feature_names=feature_cols,
    plot_type="bar",
    show=False
)
plt.title("Global Feature Importance (mean |SHAP value|)")
plt.tight_layout()
plt.show()

# Plot 2: Beeswarm — feature value vs prediction impact
# Answers: HOW does each feature's value affect the prediction?
# Red = high feature value, Blue = low feature value
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_holdout_df,
    feature_names=feature_cols,
    show=False
)
plt.title("SHAP Summary: Feature Values vs Prediction Impact")
plt.tight_layout()
plt.show()

# Top 10 features by mean |SHAP|
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_cols
).sort_values(ascending=False)

print("Top 10 features by mean |SHAP value|:")
print(mean_shap.head(10).round(5).to_string())

# ── STEP 9: SHAP — LOCAL EXPLAINABILITY ──────────────────────
# Why did the model make THIS specific prediction?
# Focus on the first wrong prediction — most instructive case

wrong_idx = np.where(tuned_preds != y_holdout.values)[0]
sample_idx = wrong_idx[0]

print(f"\nExplaining prediction for : {X_holdout.index[sample_idx].date()}")
print(f"Model predicted           : {'Up' if tuned_preds[sample_idx] == 1 else 'Down'}")
print(f"Actual outcome            : {'Up' if y_holdout.iloc[sample_idx] == 1 else 'Down'}")
print(f"Confidence (proba)        : {tuned_proba[sample_idx]:.3f}")

shap.force_plot(
    explainer.expected_value,
    shap_values[sample_idx],
    X_holdout_df.iloc[sample_idx],
    feature_names=feature_cols,
    matplotlib=True,
    show=True
)

# ── STEP 10: PRODUCTION PREDICTION PIPELINE ──────────────────
import yfinance as yf


class QuantPredictionPipeline:
    """
    Production-ready wrapper around the trained ML model.
    Given a ticker, downloads latest data, engineers features,
    and returns a trading signal with SHAP-based explanation.
    """

    def __init__(self, model, scaler, feature_cols, spy_ticker="SPY"):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.spy_ticker = spy_ticker

    def get_latest_signal(self, ticker, lookback_days=300):
        """
        Returns:
            signal      : 1 (go long) or 0 (stay out)
            confidence  : probability of an up day
            top_drivers : top 3 SHAP features driving the prediction
        """
        # Download fresh data
        end_date = pd.Timestamp.today()
        start_date = end_date - pd.Timedelta(days=lookback_days)

        raw = yf.download(ticker, start=start_date, end=end_date,
                          auto_adjust=True, progress=False)
        spy = yf.download(self.spy_ticker, start=start_date, end=end_date,
                          auto_adjust=True, progress=False)

        # Flatten MultiIndex columns if present (newer yfinance versions)
        raw.columns = raw.columns.get_level_values(0)
        spy.columns = spy.columns.get_level_values(0)

        if len(raw) < 60:
            raise ValueError(f"Not enough data for {ticker} — need at least 60 rows")

        # Engineer features using the same function as training
        featured = build_features(raw, spy)

        # Get most recent complete row
        latest = featured[self.feature_cols].dropna().iloc[[-1]]

        # Scale and predict
        latest_scaled = self.scaler.transform(latest)
        proba = self.model.predict_proba(latest_scaled)[0, 1]
        signal = int(proba > 0.5)

        # SHAP explanation for this single prediction
        exp = shap.TreeExplainer(self.model)
        shap_vals = exp.shap_values(
            pd.DataFrame(latest_scaled, columns=self.feature_cols)
        )
        top_idx = np.argsort(np.abs(shap_vals[0]))[::-1][:3]
        top_drivers = [
            (self.feature_cols[i], round(shap_vals[0][i], 4))
            for i in top_idx
        ]

        return {
            'ticker': ticker,
            'date': latest.index[0].date(),
            'signal': signal,
            'confidence': round(proba, 4),
            'action': "BUY / HOLD LONG" if signal == 1 else "STAY OUT / FLAT",
            'top_drivers': top_drivers
        }


# ── Instantiate and run ───────────────────────────────────────
quant_pipeline = QuantPredictionPipeline(
    model=best_model,
    scaler=best_scaler,
    feature_cols=feature_cols
)

result = quant_pipeline.get_latest_signal("AAPL")

print("\n" + "=" * 50)
print(f"  LIVE PREDICTION for {result['ticker']}")
print("=" * 50)
print(f"  Date       : {result['date']}")
print(f"  Signal     : {result['signal']}")
print(f"  Action     : {result['action']}")
print(f"  Confidence : {result['confidence']:.1%}")
print(f"\n  Top prediction drivers:")
for feat, shap_val in result['top_drivers']:
    direction = "▲ pushes UP" if shap_val > 0 else "▼ pushes DOWN"
    print(f"    {feat:25s}  {direction}  (SHAP: {shap_val:+.4f})")
print("=" * 50)